# RAG con BGE-M3: busqueda hibrida (densa + dispersa) en Qdrant, chunk_size=800

Version reducida del notebook principal, enfocada en UNA sola configuracion: `BAAI/bge-m3` con `chunk_size=800` (tokens, overlap 0), pero aprovechando que tanto el modelo como Qdrant soportan busqueda hibrida:

- **BGE-M3** puede generar, en una sola pasada, un vector **denso** (semantico, 1024 dim) y uno **disperso** (lexical weights por token, similar en espiritu a BM25 pero aprendido) para el mismo texto. Para esto se usa la libreria `FlagEmbedding` (no `sentence-transformers`, que solo expone el vector denso).
- **Qdrant** permite guardar ambos vectores en el mismo punto (vectores nombrados: `dense` + `sparse`) y combinarlos en la consulta con **Reciprocal Rank Fusion (RRF)** via su Query API (`prefetch` + `FusionQuery`).

La extraccion del PDF es identica a la del notebook principal (misma funcion `extraer_documento_unificado`). Lo que cambia es la representacion vectorial y la configuracion de la coleccion.

**Variables de entorno:** no se necesita `OPENROUTER_API_KEY` aqui (no se usa OpenAI/OpenRouter). `HF_TOKEN` tampoco es necesario porque `BAAI/bge-m3` no es un modelo "gated".


In [ ]:
%pip install -q pymupdf4llm langchain-text-splitters langchain-core transformers FlagEmbedding qdrant-client

## 1. Extraccion unificada del PDF

Misma logica que en el notebook principal: una sola llamada a `pymupdf4llm.to_markdown(..., page_chunks=True)`, tablas separadas del texto narrativo, y mapa de offsets -> pagina.


In [1]:
import re
import os
from typing import Dict, List, Optional, Tuple

import pymupdf4llm

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

# Ruta del PDF final a procesar
pdf_path = "./archivo_fuente/7RAAC_Ingenierias Sistemas UV-24042025.pdf"

c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PATRON_TABLA_MD = re.compile(
    r'(^\|.*\|\s*\n'
    r'(?:\|[ \t]*:?-+:?[ \t]*\|.*\n)'
    r'(?:\|.*\|\s*\n?)+)',
    re.MULTILINE,
)


def _separar_tablas_de_texto(texto: str, numero_pagina: Optional[int]) -> Tuple[str, List[Dict]]:
    tablas_encontradas = PATRON_TABLA_MD.findall(texto)
    tablas = [
        {"numero_pagina": numero_pagina, "contenido_md": t.strip()}
        for t in tablas_encontradas
    ]

    contador = {"i": 0}

    def _reemplazo(match):
        contador["i"] += 1
        return f"\n[tabla {contador['i']} de la pagina {numero_pagina}]\n"

    texto_sin_tablas = PATRON_TABLA_MD.sub(_reemplazo, texto)
    return texto_sin_tablas, tablas


def extraer_documento_unificado(
    pdf_path: str,
    table_strategy: str = "lines_strict",
    header: bool = False,
    footer: bool = False,
) -> Dict:
    paginas = pymupdf4llm.to_markdown(
        pdf_path,
        page_chunks=True,
        table_strategy=table_strategy,
        header=header,
        footer=footer,
    )

    if isinstance(paginas, str):
        raise RuntimeError(
            "pymupdf4llm.to_markdown no devolvio una lista por pagina. "
            "Revisa tu version de pymupdf4llm / si tienes pymupdf-layout instalado."
        )

    partes_texto: List[str] = []
    mapa_paginas: List[Tuple[int, int, Optional[int]]] = []
    tablas_extraidas: List[Dict] = []
    offset_actual = 0

    for pagina in paginas:
        metadata_pagina = pagina.get("metadata", {})
        numero_pagina = metadata_pagina.get("page", metadata_pagina.get("page_number"))

        texto_pagina = pagina.get("text", "")
        texto_sin_tablas, tablas_pagina = _separar_tablas_de_texto(texto_pagina, numero_pagina)
        tablas_extraidas.extend(tablas_pagina)

        if partes_texto:
            partes_texto.append(" ")
            offset_actual += 1

        inicio = offset_actual
        partes_texto.append(texto_sin_tablas)
        offset_actual += len(texto_sin_tablas)
        fin = offset_actual

        mapa_paginas.append((inicio, fin, numero_pagina))

    for i, tabla in enumerate(tablas_extraidas):
        tabla["indice"] = i

    return {
        "texto_narrativo": "".join(partes_texto),
        "mapa_paginas": mapa_paginas,
        "tablas": tablas_extraidas,
    }


def _paginas_de_rango(mapa_paginas, inicio: int, fin: int) -> List[Optional[int]]:
    paginas = [
        num for (ini_p, fin_p, num) in mapa_paginas
        if ini_p < fin and fin_p > inicio and num is not None
    ]
    return paginas or [None]

In [6]:
extraido = extraer_documento_unificado(pdf_path)

documentos_tablas = [
    Document(
        page_content=t["contenido_md"],
        metadata={
            "tipo": "tabla",
            "pagina": t["numero_pagina"],
            "fuente": pdf_path,
            "indice_tabla": t["indice"],
        },
    )
    for t in extraido["tablas"]
]

print(f"Paginas procesadas: {len(extraido['mapa_paginas'])}")
print(f"Longitud del texto narrativo: {len(extraido['texto_narrativo'])} caracteres")
print(f"Tablas extraidas: {len(documentos_tablas)}")

Paginas procesadas: 330
Longitud del texto narrativo: 649953 caracteres
Tablas extraidas: 222


## 2. Chunkeo con chunk_size fijo = 800 (tokens del tokenizer de BGE-M3)

Una sola configuracion, no un loop: `chunk_size=800`, `chunk_overlap=0`, medido con el tokenizer real de `BAAI/bge-m3`.


In [7]:
from transformers import AutoTokenizer

TOKENIZER_BGE_M3 = AutoTokenizer.from_pretrained("BAAI/bge-m3")

CHUNK_SIZE = 800
CHUNK_OVERLAP = 0


def contar_tokens_bge_m3(texto: str) -> int:
    return len(TOKENIZER_BGE_M3.encode(texto, add_special_tokens=False))


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=contar_tokens_bge_m3,
    add_start_index=True,
)
chunks = splitter.create_documents([extraido["texto_narrativo"]])

documentos_texto = []
for chunk in chunks:
    inicio = chunk.metadata.get("start_index", 0)
    fin = inicio + len(chunk.page_content)
    paginas = _paginas_de_rango(extraido["mapa_paginas"], inicio, fin)
    paginas_validas = [p for p in paginas if p is not None]

    documentos_texto.append(Document(
        page_content=chunk.page_content,
        metadata={
            "tipo": "texto",
            "pagina_inicio": min(paginas_validas) if paginas_validas else None,
            "pagina_fin": max(paginas_validas) if paginas_validas else None,
            "fuente": pdf_path,
            "chunk_size_config": CHUNK_SIZE,
            "modelo_embedding": "BAAI/bge-m3",
        },
    ))

largos = [contar_tokens_bge_m3(d.page_content) for d in documentos_texto]
promedio = sum(largos) / len(largos) if largos else 0
print(f"Chunks de texto: {len(documentos_texto)} (largo promedio real: {promedio:.0f} tokens)")

Chunks de texto: 200 (largo promedio real: 719 tokens)


## 3. BGE-M3 con salida densa + dispersa (FlagEmbedding)

`sentence-transformers` solo expone el vector denso de BGE-M3. Para el vector disperso (lexical weights) se necesita `FlagEmbedding.BGEM3FlagModel`, que calcula ambos en una sola pasada.

`embeber_hibrido` sirve tanto para documentos como para consultas: BGE-M3 es simetrico (a diferencia de `embeddinggemma`, no distingue `encode_query` de `encode_document`).


In [10]:
# Para solucinar problemas en windows
import os
import shutil
from pathlib import Path

# Fuerza a huggingface_hub a copiar archivos en vez de crear symlinks
# (evita el OSError de privilegios en Windows). Debe ir ANTES de importar
# FlagEmbedding/transformers/huggingface_hub.
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

# Limpia la descarga a medias de bge-m3 (quedo en un estado inconsistente
# cuando fallo a la mitad). Si no existe la carpeta, no pasa nada.
ruta_cache_bge = Path.home() / ".cache" / "huggingface" / "hub" / "models--BAAI--bge-m3"
if ruta_cache_bge.exists():
    shutil.rmtree(ruta_cache_bge)
    print(f"Cache incompleto borrado: {ruta_cache_bge}")

Cache incompleto borrado: C:\Users\gomex\.cache\huggingface\hub\models--BAAI--bge-m3


In [11]:
from FlagEmbedding import BGEM3FlagModel

# use_fp16=True acelera en GPU; si corres solo en CPU y da problemas, cambia a False
modelo_bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)


def embeber_hibrido(textos, tam_lote=12, max_length=8192):
    """Devuelve (vectores_densos, vectores_dispersos) para una lista de textos.
    vectores_dispersos es una lista de dicts {"indices": [...], "values": [...]},
    listos para construir un qdrant_client.models.SparseVector."""
    if not textos:
        return [], []
    salida = modelo_bge.encode(
        textos,
        batch_size=tam_lote,
        max_length=max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    densos = [v.tolist() for v in salida["dense_vecs"]]
    dispersos = [
        {"indices": [int(k) for k in lw.keys()], "values": list(lw.values())}
        for lw in salida["lexical_weights"]
    ]
    return densos, dispersos

Loading weights: 100%|██████████| 391/391 [00:01<00:00, 297.95it/s]


## 4. Coleccion Qdrant hibrida (vector denso + vector disperso)

Una sola coleccion, con dos vectores nombrados: `dense` (`VectorParams`, tamano 1024, coseno) y `sparse` (`SparseVectorParams`, sin tamano fijo).


In [13]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, SparseVectorParams, Distance, PointStruct, SparseVector,
    Prefetch, FusionQuery, Fusion,
)
import uuid

RUTA_QDRANT_HIBRIDO = "./qdrant_data_hibrido"  # carpeta propia, distinta de la del notebook principal

# Cierra el cliente anterior si ya existe en el kernel (evita el lock al re-ejecutar la celda)
try:
    client_qdrant.close()
except NameError:
    pass
except Exception as e:
    print(f"Aviso: no se pudo cerrar el cliente anterior ({e})")

client_qdrant = QdrantClient(path=RUTA_QDRANT_HIBRIDO)

COLECCION_HIBRIDA = "rag_baai_bge_m3_hibrido_cs800"
DIMENSION_DENSA = 1024

if not client_qdrant.collection_exists(COLECCION_HIBRIDA):
    client_qdrant.create_collection(
        collection_name=COLECCION_HIBRIDA,
        vectors_config={"dense": VectorParams(size=DIMENSION_DENSA, distance=Distance.COSINE)},
        sparse_vectors_config={"sparse": SparseVectorParams()},
    )
    print(f"Coleccion \'{COLECCION_HIBRIDA}\' creada.")
else:
    print(f"Coleccion \'{COLECCION_HIBRIDA}\' ya existia, se reutiliza.")

Coleccion 'rag_baai_bge_m3_hibrido_cs800' creada.


## 5. Indexar texto + tablas (denso y disperso en el mismo punto)

Igual que en el notebook principal, las tablas no se chunkean: van como un punto por tabla.


In [14]:
textos_chunks = [d.page_content for d in documentos_texto]
vectores_densos_texto, vectores_dispersos_texto = embeber_hibrido(textos_chunks)

textos_tablas = [d.page_content for d in documentos_tablas]
vectores_densos_tablas, vectores_dispersos_tablas = embeber_hibrido(textos_tablas)

puntos = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector={"dense": denso, "sparse": SparseVector(indices=disperso["indices"], values=disperso["values"])},
        payload={"page_content": doc.page_content, **doc.metadata},
    )
    for doc, denso, disperso in zip(documentos_texto, vectores_densos_texto, vectores_dispersos_texto)
]
puntos += [
    PointStruct(
        id=str(uuid.uuid4()),
        vector={"dense": denso, "sparse": SparseVector(indices=disperso["indices"], values=disperso["values"])},
        payload={"page_content": doc.page_content, **doc.metadata, "chunk_size_config": CHUNK_SIZE},
    )
    for doc, denso, disperso in zip(documentos_tablas, vectores_densos_tablas, vectores_dispersos_tablas)
]

if puntos:
    client_qdrant.upsert(collection_name=COLECCION_HIBRIDA, points=puntos)

print(f"Coleccion \'{COLECCION_HIBRIDA}\': {len(puntos)} puntos ({len(documentos_texto)} texto + {len(documentos_tablas)} tablas)")

Inference Embeddings: 100%|██████████| 19/19 [02:14<00:00,  7.06s/it]


Coleccion 'rag_baai_bge_m3_hibrido_cs800': 422 puntos (200 texto + 222 tablas)


## 6. Ejemplo rapido: consulta hibrida con fusion RRF

`prefetch` trae candidatos de cada rama (densa y dispersa) por separado, `FusionQuery(fusion=Fusion.RRF)` los combina por posicion (no por score bruto, ya que la escala de coseno y la de lexical weights no es comparable).


In [15]:
LIMITE_PREFETCH = 20  # candidatos por rama antes de fusionar


def buscar_hibrido(pregunta: str, k: int = 5):
    vectores_densos, vectores_dispersos = embeber_hibrido([pregunta])
    vector_denso = vectores_densos[0]
    vector_disperso = vectores_dispersos[0]

    resultados = client_qdrant.query_points(
        collection_name=COLECCION_HIBRIDA,
        prefetch=[
            Prefetch(query=vector_denso, using="dense", limit=LIMITE_PREFETCH),
            Prefetch(
                query=SparseVector(indices=vector_disperso["indices"], values=vector_disperso["values"]),
                using="sparse",
                limit=LIMITE_PREFETCH,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=k,
        with_payload=True,
    )
    return resultados.points


for punto in buscar_hibrido("¿qué es el PEP?"):
    pagina = punto.payload.get("pagina_inicio", punto.payload.get("pagina"))
    print(f"score_rrf={punto.score:.4f} | tipo={punto.payload.get('tipo')} | pagina={pagina}")
    print(punto.payload.get("page_content")[:300])
    print("---")

score_rrf=1.0000 | tipo=texto | pagina=38
En el PEP se define la identidad del programa y se orientan las acciones formativas en el marco misional de la Universidad del Valle. El PEP Adopta los lineamientos curriculares del <u>Acuerdo 025 del 25 de septiembre de 2015</u> (ver Anexo 27) del Consejo Superior, que actualizó la Política Curricu
---
score_rrf=0.6667 | tipo=tabla | pagina=50
|**Evaluación del Fact**|**or 1. Proyecto Educativo del Programa e Identidad Institucional**<br>|
|---|---|
|**Fortaleza del programa**|_A partir de la evaluación y análisis de este factor se consideran_<br>_fortalezas los siguientes aspectos._<br>_El Proyecto Educativo del Programa (PEP) de Ingenie
---
score_rrf=0.5000 | tipo=texto | pagina=41
En el PEP del programa de Ingeniería de Sistemas se acogen las 8 competencias definidas por la Facultad. Además, se definen 12 competencias específicas adicionales que están orientadas a formar al estudiante en los ámbitos en los que se puede ocupar como ingeniero

## 7. Retrieval completo sobre el dataset de validacion (k=5, busqueda hibrida)

Misma logica que en el notebook principal (Seccion 4), adaptada a esta unica configuracion hibrida. Cada pregunta se embebe una sola vez (denso + disperso) y se reutiliza en la consulta con RRF.

`score_rrf` en `metadatos_recuperados` **no es comparable** con los scores de coseno de las otras 9 configuraciones dense-only del notebook principal — es la posicion fusionada por RRF, no una similitud. Para comparar configuraciones usa los contextos recuperados (y metricas de RAGAS despues), no el score crudo.


In [16]:
import json
from pathlib import Path

RUTA_DATASET = "./validacion/dataset_validacion_final.json"
RUTA_SALIDA = "./validacion/resultados_retrieval_bge_m3_hibrido.json"
K = 5

with open(RUTA_DATASET, "r", encoding="utf-8") as f:
    dataset = json.load(f)

preguntas = dataset["pares"]
print(f"Preguntas cargadas: {len(preguntas)}")

Preguntas cargadas: 296


In [17]:
textos_preguntas = [p["pregunta"] for p in preguntas]
vectores_densos_preg, vectores_dispersos_preg = embeber_hibrido(textos_preguntas)

resultados_config = []
for pregunta_info, v_denso, v_disperso in zip(preguntas, vectores_densos_preg, vectores_dispersos_preg):
    hits = client_qdrant.query_points(
        collection_name=COLECCION_HIBRIDA,
        prefetch=[
            Prefetch(query=v_denso, using="dense", limit=LIMITE_PREFETCH),
            Prefetch(
                query=SparseVector(indices=v_disperso["indices"], values=v_disperso["values"]),
                using="sparse",
                limit=LIMITE_PREFETCH,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=K,
        with_payload=True,
    ).points

    resultados_config.append({
        "id": pregunta_info["id"],
        "pregunta": pregunta_info["pregunta"],
        "respuesta_esperada": pregunta_info.get("respuesta_esperada"),
        "evidencia_gold": pregunta_info.get("evidencia_gold"),
        "tipo": pregunta_info.get("tipo"),
        "dificultad": pregunta_info.get("dificultad"),
        "tipo_evidencia": pregunta_info.get("tipo_evidencia"),
        "seccion": pregunta_info.get("seccion"),
        "contextos_recuperados": [h.payload.get("page_content") for h in hits],
        "metadatos_recuperados": [
            {
                "score_rrf": h.score,
                "tipo": h.payload.get("tipo"),
                "pagina_inicio": h.payload.get("pagina_inicio", h.payload.get("pagina")),
                "pagina_fin": h.payload.get("pagina_fin"),
            }
            for h in hits
        ],
    })

salida = {
    "modelo_embedding": "BAAI/bge-m3",
    "chunk_size": CHUNK_SIZE,
    "tipo_busqueda": "hibrida_dense_sparse_rrf",
    "coleccion": COLECCION_HIBRIDA,
    "k": K,
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "resultados": resultados_config,
}

Path(RUTA_SALIDA).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA, "w", encoding="utf-8") as f:
    json.dump(salida, f, ensure_ascii=False, indent=2)

print(f"\nListo. Guardado en {RUTA_SALIDA}")

Inference Embeddings: 100%|██████████| 25/25 [00:16<00:00,  1.48it/s]



Listo. Guardado en ./validacion/resultados_retrieval_bge_m3_hibrido.json


In [18]:
# Sanity check rapido
tamano_mb = Path(RUTA_SALIDA).stat().st_size / (1024 * 1024)
n_con_contexto = sum(1 for r in resultados_config if r["contextos_recuperados"])
print(f"Archivo: {RUTA_SALIDA} ({tamano_mb:.2f} MB)")
print(f"{len(resultados_config)} preguntas, {n_con_contexto} con al menos 1 contexto recuperado")

Archivo: ./validacion/resultados_retrieval_bge_m3_hibrido.json (4.98 MB)
296 preguntas, 296 con al menos 1 contexto recuperado


In [19]:
# --- Experimento: BGE-m3 hibrido cs800, k mas alto ---
# Hipotesis: RRF fusiona el ranking denso y el disperso por posicion. En
# preguntas donde antes fallaba (recall=0), es posible que la rama dispersa
# si ubique el chunk correcto pero no lo bastante arriba como para quedar en
# el top-5 fusionado. Subir k le da mas margen, a costa de algo de precision.
COLECCION_EXP_BGE = "rag_baai_bge_m3_hibrido_cs800"  # coleccion ya indexada (ver evaluacion_ragas_resumen.csv)
K_EXP_BGE = 10
RUTA_SALIDA_EXP_BGE = f"./validacion/resultados_retrieval_bge_m3_hibrido_k{K_EXP_BGE}.json"

print(f"Experimento: BAAI/bge-m3 hibrido | coleccion '{COLECCION_EXP_BGE}' | k={K_EXP_BGE}")

vectores_densos_exp, vectores_dispersos_exp = embeber_hibrido(textos_preguntas)

resultados_exp_bge = []
for pregunta_info, v_denso, v_disperso in zip(preguntas, vectores_densos_exp, vectores_dispersos_exp):
    hits = client_qdrant.query_points(
        collection_name=COLECCION_EXP_BGE,
        prefetch=[
            Prefetch(query=v_denso, using="dense", limit=LIMITE_PREFETCH),
            Prefetch(
                query=SparseVector(indices=v_disperso["indices"], values=v_disperso["values"]),
                using="sparse",
                limit=LIMITE_PREFETCH,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=K_EXP_BGE,
        with_payload=True,
    ).points

    resultados_exp_bge.append({
        "id": pregunta_info["id"],
        "pregunta": pregunta_info["pregunta"],
        "respuesta_esperada": pregunta_info.get("respuesta_esperada"),
        "evidencia_gold": pregunta_info.get("evidencia_gold"),
        "tipo": pregunta_info.get("tipo"),
        "dificultad": pregunta_info.get("dificultad"),
        "tipo_evidencia": pregunta_info.get("tipo_evidencia"),
        "seccion": pregunta_info.get("seccion"),
        "contextos_recuperados": [h.payload.get("page_content") for h in hits],
        "metadatos_recuperados": [
            {
                "score_rrf": h.score,
                "tipo": h.payload.get("tipo"),
                "pagina_inicio": h.payload.get("pagina_inicio", h.payload.get("pagina")),
                "pagina_fin": h.payload.get("pagina_fin"),
            }
            for h in hits
        ],
    })

salida_exp_bge = {
    "modelo_embedding": "BAAI/bge-m3",
    "chunk_size": 800,
    "tipo_busqueda": "hibrida_dense_sparse_rrf",
    "coleccion": COLECCION_EXP_BGE,
    "k": K_EXP_BGE,
    "experimento": f"k subido a {K_EXP_BGE} (hipotesis: mas margen para que RRF recupere aciertos de la rama dispersa que quedan justo fuera del top-5)",
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "resultados": resultados_exp_bge,
}

Path(RUTA_SALIDA_EXP_BGE).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA_EXP_BGE, "w", encoding="utf-8") as f:
    json.dump(salida_exp_bge, f, ensure_ascii=False, indent=2)

tamano_mb = Path(RUTA_SALIDA_EXP_BGE).stat().st_size / (1024 * 1024)
n_con_contexto = sum(1 for r in resultados_exp_bge if r["contextos_recuperados"])
print(f"\nListo. Guardado en {RUTA_SALIDA_EXP_BGE} ({tamano_mb:.2f} MB)")
print(f"{len(resultados_exp_bge)} preguntas, {n_con_contexto} con al menos 1 contexto recuperado")

Experimento: BAAI/bge-m3 hibrido | coleccion 'rag_baai_bge_m3_hibrido_cs800' | k=10


Inference Embeddings: 100%|██████████| 25/25 [00:15<00:00,  1.59it/s]



Listo. Guardado en ./validacion/resultados_retrieval_bge_m3_hibrido_k10.json (9.53 MB)
296 preguntas, 296 con al menos 1 contexto recuperado


## Notas finales

- Esta coleccion (`rag_baai_bge_m3_hibrido_cs800`) y su archivo de resultados (`resultados_retrieval_bge_m3_hibrido.json`) son un **10mo punto** en tu matriz de pruebas, junto a las 9 configuraciones dense-only del notebook principal.
- `RUTA_QDRANT_HIBRIDO` usa una carpeta propia (`./qdrant_data_hibrido`) para no chocar con el lock del `client_qdrant` del notebook principal si lo tienes abierto en otro kernel.
- Si quisieras ademas el vector multi-token (ColBERT/late-interaction) de BGE-M3, se activa con `return_colbert_vecs=True` en `embeber_hibrido` y se guarda en Qdrant con `MultiVectorConfig` — no incluido aqui porque solo pediste denso + disperso.
- Para ajustar el peso relativo de RRF entre las dos ramas, Qdrant (>=1.16) admite `query={"rrf": {"k": 60}}` (o `models.RrfQuery`) en vez de `FusionQuery(fusion=Fusion.RRF)`, si tu version de `qdrant-client` lo soporta.
